# Approximate Bayesian Computation on a Lotka-Volterra model

Supporting analysis for the ABC class presentation. Everything the deck quotes is
produced here.

**What ABC is for.** Bayesian inference needs `p(y | theta)` as a number you can
evaluate. For many mechanistic models that number is an integral over every
unobserved path and cannot be computed, even though the model is trivial to
simulate forward. ABC replaces *evaluating* the likelihood with *sampling* from
it: propose parameters, simulate, and keep the proposal if the simulated data
lands within a tolerance `eps` of the real data.

**Why Lotka-Volterra.** You would not actually need ABC here, and that is the
point. The ODE is deterministic, so with Gaussian observation noise the
likelihood is available in closed form and the exact posterior can be computed on
a grid. That makes this a case where ABC can be *checked* rather than merely
trusted.

In [ ]:
import numpy as np
import pandas as pd

from lotka_volterra import lv_rk4, make_data, noise_floor
from abc_inference import (rejection_abc, prior_sample_distances,
                           exact_posterior_grid, grid_moments, abc_smc,
                           distances)

A_TRUE = B_TRUE = 1.0
T_OBS = np.arange(1.0, 15.1, 1.0)      # 15 observation times
SIGMAS = [0.1, 0.5, 1.0]
N_OBS = 2 * len(T_OBS)                 # 2 species -> 30 data points

truth, data = make_data(SIGMAS, T_OBS)
print("observations per dataset:", N_OBS)

## 1. The model and the synthetic data

    dx/dt = a*x - x*y      prey
    dy/dt = b*x*y - y      predator

`a` sets how fast prey recover; `b` sets how efficiently predators convert prey
into more predators. The `x*y` term is an encounter rate: a meeting needs one of
each, so it scales with the product rather than the sum.

Truth is `a = b = 1`, `x0 = 1`, `y0 = 0.5`, giving about two full cycles over
`t = 0..15`. Each dataset is the true trajectory plus i.i.d. Gaussian noise.

In [ ]:
# Tolerance rule: the residual norm we EXPECT from the noise model alone.
# Fixing eps this way means it is chosen before any output is seen, rather than
# tuned until the answer looks good.
EPS = {s: noise_floor(s, N_OBS) for s in SIGMAS}
FLOOR = {s: float(np.linalg.norm(data[s] - truth)) for s in SIGMAS}

pd.DataFrame({"sigma": SIGMAS,
              "eps = sigma*sqrt(2n)": [EPS[s] for s in SIGMAS],
              "realised floor ||e_obs||": [FLOOR[s] for s in SIGMAS]}).round(3)

## 2. Rejection ABC

Draw `(a, b)` from a uniform prior on `(0, 5)`, solve the ODE, and keep the draw
if the trajectory is within `eps` of the data. The distance is plain Euclidean
over all 30 observations — no summary statistics, because 30 numbers is already
low-dimensional and compressing them would throw away phase information.

All three noise levels are scored off the same set of simulations, which is what
makes a 6-million-draw budget affordable. This cell takes a couple of minutes.

In [ ]:
abc, n_total = rejection_abc(data, T_OBS, EPS, n_sim=6_000_000)

pd.DataFrame([{"sigma": s, "eps": round(EPS[s], 3), "accepted": abc[s][0].size,
               "acceptance %": round(100 * abc[s][0].size / n_total, 4),
               "mean a": round(abc[s][0].mean(), 4), "sd a": round(abc[s][0].std(), 4),
               "mean b": round(abc[s][1].mean(), 4), "sd b": round(abc[s][1].std(), 4)}
              for s in SIGMAS])

## 3. Checking against the exact posterior

Because the simulator is deterministic, `log p(y | theta)` is just the scaled
negative sum of squared residuals. Evaluating it on a 260 x 260 grid gives a
reference with no Monte Carlo error at all. The edge mass is reported as a
sanity check that the grid is wide enough to hold essentially all the mass.

In [ ]:
exact = {}
rows = []
for s in SIGMAS:
    A, B, P = exact_posterior_grid(data[s], s, T_OBS)
    m = grid_moments(A, B, P)
    exact[s] = (A, B, P, m)
    edge = P[0, :].sum() + P[-1, :].sum() + P[:, 0].sum() + P[:, -1].sum()
    rows.append({"sigma": s,
                 "ABC a": f"{abc[s][0].mean():.3f} +/- {abc[s][0].std():.3f}",
                 "exact a": f"{m[0]:.3f} +/- {m[1]:.3f}",
                 "ABC b": f"{abc[s][1].mean():.3f} +/- {abc[s][1].std():.3f}",
                 "exact b": f"{m[2]:.3f} +/- {m[3]:.3f}",
                 "sd ratio": round(abc[s][0].std() / m[1], 2),
                 "edge mass": f"{edge:.1e}"})
pd.DataFrame(rows)

ABC lands on the right centre and the right `a`-`b` tilt at every noise level,
and is over-dispersed by roughly 1.5x. That direction is expected: a hard
tolerance ball is a blunter kernel than the Gaussian likelihood it stands in for.

Note that at `sigma = 1.0` the *exact* posterior is itself off-centre in `b`.
That is this particular draw of noise, not a failure of ABC — approximation error
and sampling error are different things.

## 4. The tolerance is the whole game

Retaining distances from a prior sample lets the tolerance be re-thresholded
without re-simulating, which is how the sweep below is produced. Two failure
modes bracket the useful range:

- `eps` too large: the acceptance region covers most of the prior, so the
  posterior reverts to the prior.
- `eps` too small: with a deterministic simulator the ABC posterior is the prior
  *truncated* to `{theta : rho(theta) <= eps}`, and that set collapses onto the
  single best-fitting parameter for this one noise realisation — falsely precise.

The second point is the one worth remembering: ABC does **not** automatically
converge to the Bayesian posterior as `eps -> 0` under a hard indicator. To
recover it you either put the observation noise inside the simulator, or replace
the indicator with a smooth kernel `exp(-rho^2 / 2 sigma^2)` — which is the
likelihood again.

In [ ]:
a_s, b_s, dist = prior_sample_distances(data, T_OBS, n_sim=300_000)

S = 0.5
sweep = []
for e in [14, 10, 8, 6, 4, 3.0, EPS[S], FLOOR[S] * 1.01]:
    m = dist[S] <= e
    sweep.append({"eps": round(e, 3), "acceptance %": round(100 * m.mean(), 3),
                  "n": int(m.sum()),
                  "sd a": round(a_s[m].std(), 3) if m.sum() > 5 else None,
                  "sd b": round(b_s[m].std(), 3) if m.sum() > 5 else None})
print(f"exact posterior sd at sigma={S}: a={exact[S][3][1]:.3f}, b={exact[S][3][3]:.3f}")
print(f"prior sd for reference: {np.std(np.random.default_rng(0).uniform(0, 5, 100000)):.3f}")
pd.DataFrame(sweep)

## 5. ABC-SMC

Rejection ABC keeps proposing from the prior, so almost every simulation is
wasted. ABC-SMC walks a decreasing tolerance schedule, each population reusing
the last: resample by weight, perturb with a Gaussian kernel of twice the
population variance, and reweight to undo the fact that proposals no longer come
from the prior.

In [ ]:
EPS_SCHEDULE = [12.0, 8.0, 5.0, 3.6, 3.0, EPS[0.5]]
particles, weights, report = abc_smc(data[0.5], T_OBS, EPS_SCHEDULE)

smc_total = sum(r["simulations"] for r in report)
rej_rate = (dist[0.5] <= EPS_SCHEDULE[-1]).mean()
rej_equiv = int(len(particles[0]) / rej_rate)
print(f"ABC-SMC simulations: {smc_total:,}")
print(f"Rejection ABC for the same {len(particles[0])} particles: {rej_equiv:,}")
print(f"speed-up: {rej_equiv / smc_total:.1f}x")
print(f"SMC posterior: a={np.average(particles[0], weights=weights):.3f}, "
      f"b={np.average(particles[1], weights=weights):.3f}")
pd.DataFrame(report).round(4)

Both samplers land on the same posterior. The sampler changes the cost, not
the target — a useful thing to be clear about, since it is easy to imply that
SMC is somehow more accurate rather than just cheaper.

## 6. Figures

In [ ]:
from abc_figures import fig_data, fig_posterior, fig_tolerance

fig_data(truth, data, abc, SIGMAS, T_OBS, EPS)
fig_posterior(abc, exact, SIGMAS)
fig_tolerance(a_s, b_s, dist[0.5], 0.5, FLOOR[0.5], EPS[0.5],
              (exact[0.5][3][1], exact[0.5][3][3]))
print("wrote fig_lv_data.png, fig_lv_posterior.png, fig_lv_tolerance.png")

## Caveats

- `sigma` is treated as **known** when setting `eps`. Inferring it jointly is the
  honest version and would widen the posteriors.
- The tolerance kernel is a hard indicator, not a smooth kernel, which is why the
  ABC posteriors are over-dispersed relative to exact.
- Plain Lotka-Volterra has no prey carrying capacity, so with `a = b = 1` the
  cycle is neutrally stable and its amplitude is set entirely by the initial
  conditions. Fine for demonstrating an inference method; not a defensible
  population model.
- Solver accuracy is part of the model. RK4 at `dt = 0.01` against observation
  spacing of 1.0 keeps integration error far below `eps`; a looser solver would
  mean fitting numerical error.